In [4]:
!pip install transformers torchvision pillow scikit-learn -q


In [8]:
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
from transformers import DistilBertTokenizer, DistilBertModel
from sklearn.model_selection import train_test_split
import numpy as np
from PIL import Image
import requests
from io import BytesIO
import pandas as pd
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using: {device}")

Using: cpu


In [9]:
ads = pd.read_csv('ads_enriched.csv')
events = pd.read_csv('ad_events.csv')
campaigns = pd.read_csv('campaigns.csv')
users = pd.read_csv('users.csv')

In [12]:
#building engagement score again
weights = {'Impression ': 0, 'Click': 1,'Like': 2,'Comment': 3,'Share': 4,'Purchase': 5}
events['engagement_weight'] = events['event_type'].map(weights)

engagement = (
    events.groupby(['ad_id','user_id'])['engagement_weight'].sum()
    .reset_index().rename(columns = { 'engagement_weight': 'engagement_score' })
)

print(f"Total pairs: {len(engagement):,}")
print(engagement['engagement_score'].value_counts().head())

Total pairs: 206,789
engagement_score
0.0    174082
1.0     21549
2.0      6624
3.0      2316
5.0      1108
Name: count, dtype: int64


In [18]:
#merging and filtering

df = engagement.merge(ads, on='ad_id')
df = df.merge(campaigns[['campaign_id', 'total_budget', 'duration_days']], on='campaign_id')
df = df.merge(users[['user_id', 'user_age', 'user_gender']], on = 'user_id')
df = df[df['engagement_score']>0].copy().reset_index(drop=True)

print(f"Engaged rows: {len(df):,}")
print(f"Score range: {df['engagement_score'].min()} - {df['engagement_score'].max()}")

Engaged rows: 33,033
Score range: 1.0 - 8.0
